In [5]:
import pandas as pd
#Đọc dữ liệu fale đã được chuẩn hoá
stu_clean = pd.read_csv("data_clean/student_class_clean.csv")
atten_clean = pd.read_csv("data_clean/attendance_log_clean.csv")
classes_clean = pd.read_csv("data_clean/class_schedule_clean.csv")

In [6]:
#Kết hợp bảng student_class với attendance_log theo student_id
merged_1 = pd.merge(stu_clean, atten_clean, 
                    on='student_id', 
                    how='outer',  
                    indicator=True)
print(f"Số dòng sau merge: {len(merged_1)}")
print("\nKiểm tra dữ liệu không khớp:")
print(merged_1['_merge'].value_counts())
# Tìm sinh viên không có trong attendance_log
missing_in_log = merged_1[merged_1['_merge'] == 'left_only']
if not missing_in_log.empty:
    print(f"\nSinh viên không có trong attendance_log: {missing_in_log['student_id'].tolist()}")

Số dòng sau merge: 5

Kiểm tra dữ liệu không khớp:
_merge
left_only     2
both          2
right_only    1
Name: count, dtype: int64

Sinh viên không có trong attendance_log: ['sv003', 'sv004']


In [7]:
#Kết hợp attendance_log với class_schedule theo class_id
classes_clean['class_id_norm'] = classes_clean['class_id']
atten_clean['class_id_norm'] = atten_clean['class_id']
merged_final = pd.merge(merged_1, classes_clean,
                        left_on='class_id_norm',
                        right_on='class_id_norm',
                        how='outer',
                        suffixes=('_att', '_class'),
                        indicator='merge2')
print(f"Số dòng sau merge 2: {len(merged_final)}")
print("\nKiểm tra dữ liệu không khớp ở merge 2:")
print(merged_final['merge2'].value_counts())

KeyError: 'class_id_norm'

In [ ]:
#Kiểm tra bất thường
# a) Class_id không khớp
class_mismatch = merged_final[merged_final['merge2'] != 'both']
if not class_mismatch.empty:
    print("1. Class_id không khớp hoặc thiếu:")
    print(class_mismatch[['class_id_att', 'class_id_norm']].drop_duplicates())

# b) Sinh viên không có lớp
no_class_students = merged_final[(merged_final['_merge'] == 'both') & 
                                 (merged_final['merge2'] == 'left_only')]
if not no_class_students.empty:
    print("\n2. Sinh viên có điểm danh nhưng không có trong class_schedule:")
    print(no_class_students[['student_id', 'class_id_att']].drop_duplicates())

# c) Lớp không có sinh viên điểm danh
no_attendance_class = merged_final[merged_final['merge2'] == 'right_only']
if not no_attendance_class.empty:
    print("\n3. Lớp trong schedule nhưng không có ai điểm danh:")
    print(no_attendance_class[['class_id_class', 'course_name']].drop_duplicates())

In [ ]:
#Xuất file
complete_data = merged_final[
    (merged_final['_merge'] == 'both') & 
    (merged_final['merge2'] == 'both')
]

# Chọn và đổi tên cột cần thiết
complete_data = complete_data[[
    'date', 'student_id', 'full_name', 'class_cohort',
    'class_id_class', 'course_name', 'weekday', 'slot',
    'status', 'note'
]].rename(columns={'class_id_class': 'class_id'})



# Xuất file CSV
complete_data.to_csv('data_clean/merge.csv', index=False, encoding='utf-8-sig')
print("\nĐã xuất file: merge.csv")
